# Week 39, Day 2: Where is that sound? Interaural time and coincidence detection

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Measuring the delay

Write estimate_itd(left, right, max_lag): compute the cross-correlation sum(left[t] * right[t + k]) for integer lags k from -max_lag to max_lag samples, and return the lag with the largest value converted to seconds. Positive values mean the right ear lags.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1948)
fs = 96000
dur = 0.1
src = r.normal(size=int(dur * fs) + 200)

def binaural(src, itd, fs):
    d = int(round(itd * fs))
    return src[100:100 + int(dur * fs)], src[100 - d:100 - d + int(dur * fs)]

def estimate_itd(left, right, max_lag):
    lags = np.arange(-max_lag, max_lag + 1)
    cc = ...
    return lags[np.argmax(cc)] / fs, lags, cc

left, right = binaural(src, 300e-6, fs)
itd_hat, lags, cc = estimate_itd(left, right, int(0.0008 * fs))
noisy_l, noisy_r = left + r.normal(size=left.size), right + r.normal(size=right.size)
itd_noisy, _, cc_noisy = estimate_itd(noisy_l, noisy_r, int(0.0008 * fs))
print(f"estimated ITD {itd_hat * 1e6:.0f} us (clean), {itd_noisy * 1e6:.0f} us (0 dB noise)")
plt.plot(lags / fs * 1e6, cc / cc.max(), label="clean"); plt.plot(lags / fs * 1e6, cc_noisy / cc_noisy.max(), label="noisy")
plt.axvline(300, color="k", ls="--"); plt.xlabel("internal delay (microseconds)"); plt.ylabel("coincidence (normalised)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_itd():
    "the estimated ITD is within 25 microseconds of the true 300"
    assert abs(itd_hat - 300e-6) < 25e-6

def test_noise():
    "with independent noise added at each ear (signal-to-noise ratio 0 dB), the estimate is still within 50 microseconds"
    assert abs(itd_noisy - 300e-6) < 50e-6

test_itd()
test_noise()
print("All checks passed")

### Exercise: Azimuth from ITD

Write itd_of(theta_deg) with Woodworth's formula, and azimuth_of(itd) inverting it numerically on a fine grid of azimuths from 0 to 90 degrees. Then convert the estimated ITD from the last cell to an azimuth.

In [ ]:
a, c = 0.0875, 343.0

def itd_of(theta_deg):
    th = np.deg2rad(theta_deg)
    return ...

def azimuth_of(itd):
    grid = np.linspace(0, 90, 9001)
    return ...

print(f"maximum ITD {itd_of(90) * 1e6:.0f} us; the ITD of {itd_hat * 1e6:.0f} us corresponds to {azimuth_of(itd_hat):.1f} degrees")
th = np.linspace(0, 90, 91)
plt.plot(th, itd_of(th) * 1e6); plt.xlabel("azimuth (deg)"); plt.ylabel("ITD (microseconds)"); plt.show()

In [ ]:
import numpy as np
def test_max():
    "the maximum ITD (at 90 degrees) is between 600 and 700 microseconds"
    assert 600e-6 < itd_of(90) < 700e-6

def test_invert():
    "azimuth_of recovers 35 degrees from its ITD to within 0.5 degrees"
    assert abs(azimuth_of(itd_of(35)) - 35) < 0.5

test_max()
test_invert()
print("All checks passed")

### Exercise: Phase ambiguity for tones

Write count_peaks(cc): the number of local maxima of the cross-correlation within 95% of its largest value. Count them for tones of 500 and 2000 Hz with an ITD of 300 microseconds, over lags of plus or minus 700 microseconds.

In [ ]:
def tone_pair(freq, itd):
    t = np.arange(int(dur * fs)) / fs
    return np.sin(2 * np.pi * freq * t), np.sin(2 * np.pi * freq * (t - itd))

def count_peaks(cc):
    ...

n_peaks = {}
for freq in (500, 2000):
    l, rr = tone_pair(freq, 300e-6)
    _, lg, ccf = estimate_itd(l, rr, int(0.0007 * fs))
    n_peaks[freq] = count_peaks(ccf)
    plt.plot(lg / fs * 1e6, ccf / ccf.max(), label=f"{freq} Hz")
print("peaks within 95% of the maximum:", n_peaks)
plt.axvline(300, color="k", ls="--"); plt.xlabel("internal delay (microseconds)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_low():
    "a 500 Hz tone has a single cross-correlation peak within the physiological range"
    assert n_peaks[500] == 1

def test_high():
    "a 2000 Hz tone has at least two peaks of nearly equal height within the range"
    assert n_peaks[2000] >= 2

test_low()
test_high()
print("All checks passed")